'''
Milestone 3

Nama  : Raka Ardhi
Batch : CODA-001-RMT

### Program ini dibuat untuk melakukan Pra Automasi menggunakan Dataset IBM HR dari  Kaggle
'''

In [1]:
!pip install -q "great-expectations==0.18.19"

In [30]:
import pandas as pd
import numpy as np
import great_expectations as gx

df = pd.read_csv('HR_IBM_Dataset.csv')
df.head()

print(df.shape)
df.info()
df.describe()

# Cek missing values
print(df.isnull().sum())

# Cek duplicates
print("\nJumlah Baris Duplikat:", df.duplicated().sum())

df['Attrition'].unique()
df['BusinessTravel'].unique()
df['Department'].unique()

# Drop kolom yang tidak relevan
df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'])

(1470, 35)
<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeCount             1470 non-null   int64
 9   EmployeeNumber            1470 non-null   int64
 10  EnvironmentSatisfaction   1470 non-null   int64
 11  Gender                    1470 non-null   str  
 12  HourlyRate                1470 non-null   int64
 13  JobInvolvement            1470 non-null   int64
 14  JobLevel                  1470 non-null 

- Kesimpulan
1. Dataset ini terdiri dari 1470 baris dan 35 kolom
2. Tidak ada missing values
3. Tidak ada duplicate yang signifikan
4. Kolom EmployeeNumber bersifat unik
5. Kolom Attrition hanya memiliki nilai Yes dan No
6. Kolom Age berada dalam rentang normal( kurang lebih 18-60)
7. Kolom EmployeeCount,StandardHours dan Over18 akan dihapus karena tidak relevan

In [32]:
# Skenario:
# employee_profile_id digunakan sebagai unique global ID HR system

df['employee_profile_id'] = (
    df['EmployeeNumber'].astype(str) + '_' +
    df['Department'] + '_' +
    df['JobRole']
)
df.to_csv('HR_IBM_Dataset_CLEAN.csv', index=False)

### Great Expectation

- Create Data context

In [33]:
from great_expectations.data_context import FileDataContext

context = FileDataContext.create(project_root_dir='./')

In [35]:
# Give a name to a Datasource. This name must be unique between Datasources.
datasource_name = 'DATA-HR'
datasource = context.sources.add_pandas(datasource_name)

# Give a name to a data asset
asset_name = 'hr-employee'
path_to_data = 'HR_IBM_Dataset_CLEAN.csv'
asset = datasource.add_csv_asset(
    name=asset_name,
    filepath_or_buffer=path_to_data
)

# Build batch request
batch_request = asset.build_batch_request()

In [36]:
expectation_suite_name = 'expectation-hr-dataset'

context.add_or_update_expectation_suite(
    expectation_suite_name=expectation_suite_name
)

validator = context.get_validator(
    batch_request=batch_request,
    expectation_suite_name=expectation_suite_name
)

validator.head()

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager,employee_profile_id
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,2,...,1,0,8,0,1,6,4,0,5,1_Sales_Sales Executive
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,2,3,...,4,1,10,3,3,10,7,1,7,2_Research & Development_Research Scientist
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,4,...,2,0,7,3,3,0,0,0,0,4_Research & Development_Laboratory Technician
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,5,4,...,3,0,8,3,3,8,7,3,0,5_Research & Development_Research Scientist
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,7,1,...,4,1,6,3,3,2,2,2,2,7_Research & Development_Laboratory Technician


1. Unique

In [ ]:
df['employee_profile_id'] = (
    df['EmployeeNumber'].astype(str) + '_' +
    df['Department'] + '_' +
    df['JobRole']
)
validator.expect_column_values_to_be_unique('EmployeeNumber')

# simpan ulang ke CSV sementara
df.to_csv('HR_IBM_Dataset_CLEAN.csv', index=False)

2. BETWEEN (Age)

In [38]:
validator.expect_column_values_to_be_between(
    column='Age',
    min_value=18,
    max_value=60
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

3. IN SET (Attrition)

In [39]:
validator.expect_column_values_to_be_in_set(
    'Attrition',
    ['Yes', 'No']
)

Calculating Metrics:   0%|          | 0/8 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": [],
    "missing_count": 0,
    "missing_percent": 0.0,
    "unexpected_percent_total": 0.0,
    "unexpected_percent_nonmissing": 0.0
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

4. TYPE LIST(MonthlyIncome)

In [40]:
validator.expect_column_values_to_be_in_type_list(
    'MonthlyIncome',
    ['int64', 'float64']
)

Calculating Metrics:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": "int64"
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

5. NOT NULL

In [41]:
validator.expect_column_values_to_not_be_null('JobRole')

Calculating Metrics:   0%|          | 0/6 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "element_count": 1470,
    "unexpected_count": 0,
    "unexpected_percent": 0.0,
    "partial_unexpected_list": []
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

6. MOST COMMON VALUE

In [42]:
validator.expect_column_most_common_value_to_be_in_set(
    column='Gender',
    value_set=['Male', 'Female']
)

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": [
      "Male"
    ]
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

7. COUNT TO BE

In [43]:
validator.expect_column_distinct_values_to_be_in_set(
    column='Department',
    value_set=['Sales', 'Research & Development', 'Human Resources']
)

Calculating Metrics:   0%|          | 0/4 [00:00<?, ?it/s]

{
  "success": true,
  "result": {
    "observed_value": [
      "Human Resources",
      "Research & Development",
      "Sales"
    ],
    "details": {
      "value_counts": [
        {
          "value": "Human Resources",
          "count": 63
        },
        {
          "value": "Research & Development",
          "count": 961
        },
        {
          "value": "Sales",
          "count": 446
        }
      ]
    }
  },
  "meta": {},
  "exception_info": {
    "raised_exception": false,
    "exception_traceback": null,
    "exception_message": null
  }
}

In [44]:
# Save into Expectation Suite

validator.save_expectation_suite(discard_failed_expectations=False)

Checkpoint

In [45]:
# Create a checkpoint

checkpoint_1 = context.add_or_update_checkpoint(
    name = 'checkpoint_1',
    validator = validator,
)

In [46]:
# Run a checkpoint

checkpoint_result = checkpoint_1.run()

Calculating Metrics:   0%|          | 0/24 [00:00<?, ?it/s]

In [47]:
# Build data docs

context.build_data_docs()

{'local_site': 'file://c:\\Users\\Mark11\\Coda_15\\Coda_15_Git_Mark\\p2-coda015-rmt-m3-markbilly11\\gx\\uncommitted/data_docs/local_site/index.html'}